# **Scraping Planos WKVE Telecom**

_Página do provedor: https://www.wkve.com.br/cidades/conselheiro-pena/#planos_

In [ ]:
# Bibliotecas
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date
import re

## **1. Extração**

In [ ]:
# Obter lista de planos
texto_html = open("../../data/scraped/wkve.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
container_planos = soup.find("div", class_="lista-planos")
planos = container_planos.find_all("div", class_="conteudo")
print(f"{len(planos)} cartões de planos capturados.")

In [ ]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find("div", class_="quantidade").get_text(strip=True)
    preco = plano.find("div", class_="bloco-p").get_text(strip=True)
    svas = plano.find("img").get("data-src")

    return {
        "velocidade": velocidade or None,
        "preco": preco or None,
        "svas": svas
    }

df = pd.DataFrame(raspar_plano(p) for p in planos).drop_duplicates().reset_index(drop=True)
df

## **2. Tratamento**

In [ ]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "WKVE"

# Velocidade
df_tratado["VELOCIDADE"] = (
    df_tratado["velocidade"]
    .astype(str)
    .str.extract(r"(\d+)\s*(?=mega|mb)", flags=re.IGNORECASE)[0]
    .astype("Int64")
)

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

# SVAs
df_tratado["SVAs"] = df_tratado["svas"].str.extract(r"_([^_]+)\.[^.]+$")

lista = ["watch", "globoplay", "apple", "prime", "hbo", "premiere", "disney"]
padrao = re.compile("|".join(lista), flags=re.IGNORECASE)
def processar(texto):
    if pd.isna(texto):
        return texto
    achados = padrao.findall(texto)
    resto = padrao.sub("", texto).strip(" _-")
    return " ou ".join(achados) + (f" ou {resto}" if resto else "") if achados else resto

df_tratado["SVAs"] = df_tratado["SVAs"].apply(processar)

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

## **3. Carga**

In [ ]:
# Salvando
provedor = "WKVE.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")